In [1]:
!pip install -q transformers datasets sentencepiece accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("rasheduzzaman/Bangla_question_answer_pair_70K_dataset")

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

final_json_formate_84K.json:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/81072 [00:00<?, ? examples/s]

In [3]:
print(dataset)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 81072
    })
})
{'input': '\nকোন আইনের অধীনে রাজশাহী সিটি কর্পোরেশন গঠিত হয়েছে?\n', 'output': '\nরাজশাহী সিটি কর্পোরেশন আইন, ১৯৮৭\n\n'}


In [4]:
dataset = dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 72964
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 8108
    })
})


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [6]:
def preprocess_function(examples):

    inputs = [
        "question: " + str(x)
        for x in examples["input"]
    ]

    targets = [
        str(x)
        for x in examples["output"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [7]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/72964 [00:00<?, ? examples/s]

Map:   0%|          | 0/8108 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 72964
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8108
    })
})


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qa-mt5",

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,

    logging_steps=100,

    learning_rate=5e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=True,

    save_total_limit=2,

    load_best_model_at_end=True,

    report_to="none"
)

In [12]:
from transformers import DataCollatorForSeq2Seq
from transformers import Trainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],

    processing_class=tokenizer,
    data_collator=data_collator
)

In [13]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("./qa-mt5-final")
tokenizer.save_pretrained("./qa-mt5-final")

In [ ]:
from transformers import pipeline

qa_model = pipeline(
    "text2text-generation",
    model="./qa-mt5-final",
    tokenizer="./qa-mt5-final"
)

In [ ]:
question = "বাংলাদেশের রাজধানী কী?"

result = qa_model(
    "question: " + question,
    max_new_tokens=64
)

print(result[0]["generated_text"])

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

def ask_question(question):

    prompt = "question: " + question

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
print(ask_question("বাংলাদেশের রাজধানী কী?"))

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
repo_id = "rakib730/bangla-qa-mt5"

In [ ]:
trainer.push_to_hub(
    repo_id=repo_id
)

In [ ]:
tokenizer.push_to_hub(repo_id)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="rakib730/bangla-qa-mt5",
    exist_ok=True
)

api.upload_folder(
    folder_path="./bangla-qa-mt5",
    repo_id="rakib730/bangla-qa-mt5"
)